In [0]:
CREATE OR REPLACE TABLE production.supply_analytics.dst_supply_dataset AS

-- ============================================
-- PARAMETERS - UPDATE THESE
-- ============================================
WITH params AS (
  SELECT 
    DATE '2025-01-01' AS yoy_pre_start_date,
    DATE '2025-01-15' AS yoy_pre_end_date,
    DATE '2025-01-16' AS yoy_event_week_start,
    DATE '2025-01-30' AS yoy_post_end_date,
    DATE '2026-01-01' AS event_pre_start_date,
    DATE '2026-01-15' AS event_pre_end_date,
    DATE '2026-01-16' AS event_week_start_str,
    DATE '2026-01-30' AS event_post_end_date,
    ARRAY('Germany') AS countries,
    ARRAY() AS tour_ids
),

event_data AS (
  SELECT
    CASE 
      WHEN h.date_id BETWEEN params.yoy_pre_start_date AND params.yoy_pre_end_date THEN 'PRE_LY'
      WHEN h.date_id BETWEEN params.yoy_event_week_start AND params.yoy_post_end_date THEN 'POST_LY'
      WHEN h.date_id BETWEEN params.event_pre_start_date AND params.event_pre_end_date THEN 'PRE_CY'
      WHEN h.date_id BETWEEN params.event_week_start_str AND params.event_post_end_date THEN 'POST_CY'
    END AS period_window,
    dl.country_name,
    h.tour_id,
    h.supplier_id,
    h.date_id,
    date_trunc('week', h.date_id) AS week_start,
    CAST(h.is_online AS INT) AS is_online
  FROM production.supply.fact_tour_review_history h
  INNER JOIN dwh.dim_location dl ON h.location_id = dl.location_id
  CROSS JOIN params
  WHERE (
      h.date_id BETWEEN params.event_pre_start_date AND params.event_post_end_date
      OR h.date_id BETWEEN params.yoy_pre_start_date AND params.yoy_post_end_date
    )
    AND (SIZE(params.tour_ids) = 0 OR ARRAY_CONTAINS(params.tour_ids, h.tour_id))
    AND (SIZE(params.countries) = 0 OR ARRAY_CONTAINS(params.countries, dl.country_name))
),

-- ============================================
-- Days online per tour per week
-- ============================================
tour_week_online AS (
  SELECT
    period_window,
    country_name,
    tour_id,
    week_start,
    SUM(is_online) AS days_online
  FROM event_data
  WHERE period_window IS NOT NULL
  GROUP BY period_window, country_name, tour_id, week_start
),

avg_tour_online AS (
  SELECT
    period_window,
    country_name,
    AVG(days_online) AS avg_days_online_per_tour_per_week
  FROM tour_week_online
  GROUP BY period_window, country_name
),

-- ============================================
-- Days online per supplier per week
-- A supplier counts as online on a day if
-- at least one of their tours is online
-- ============================================
supplier_day_online AS (
  SELECT
    period_window,
    country_name,
    supplier_id,
    week_start,
    date_id,
    MAX(is_online) AS is_online_day
  FROM event_data
  WHERE period_window IS NOT NULL
  GROUP BY period_window, country_name, supplier_id, week_start, date_id
),

supplier_week_online AS (
  SELECT
    period_window,
    country_name,
    supplier_id,
    week_start,
    SUM(is_online_day) AS days_online
  FROM supplier_day_online
  GROUP BY period_window, country_name, supplier_id, week_start
),

avg_supplier_online AS (
  SELECT
    period_window,
    country_name,
    AVG(days_online) AS avg_days_online_per_supplier_per_week
  FROM supplier_week_online
  GROUP BY period_window, country_name
),

-- ============================================
-- Main aggregation (existing metrics)
-- ============================================
main_agg AS (
  SELECT
    period_window,
    country_name,
    COUNT(DISTINCT week_start) AS n_weeks,
    COUNT(DISTINCT tour_id) AS total_tours,
    COUNT(DISTINCT CASE WHEN is_online = 1 THEN tour_id END) AS active_tours,
    CAST(COUNT(DISTINCT CASE WHEN is_online = 1 THEN tour_id END) AS DOUBLE) / NULLIF(COUNT(DISTINCT tour_id), 0) AS share_active_tours,
    COUNT(DISTINCT supplier_id) AS total_suppliers,
    COUNT(DISTINCT CASE WHEN is_online = 1 THEN supplier_id END) AS active_suppliers,
    CAST(COUNT(DISTINCT CASE WHEN is_online = 1 THEN supplier_id END) AS DOUBLE) / NULLIF(COUNT(DISTINCT supplier_id), 0) AS share_active_suppliers
  FROM event_data
  WHERE period_window IS NOT NULL
  GROUP BY period_window, country_name
)

-- ============================================
-- Final output: join everything together
-- ============================================
SELECT
  m.*,
  t.avg_days_online_per_tour_per_week,
  s.avg_days_online_per_supplier_per_week
FROM main_agg m
LEFT JOIN avg_tour_online t 
  ON m.period_window = t.period_window 
  AND m.country_name = t.country_name
LEFT JOIN avg_supplier_online s 
  ON m.period_window = s.period_window 
  AND m.country_name = s.country_name
ORDER BY m.country_name, 
  CASE m.period_window 
    WHEN 'PRE_LY' THEN 1 
    WHEN 'POST_LY' THEN 2 
    WHEN 'PRE_CY' THEN 3 
    WHEN 'POST_CY' THEN 4 
  END
